In [ ]:
from __future__ import annotations

import traceback
import numpy as np
import pandas as pd
from IPython.display import display

from battery_deg_spme.config.settings import get_default_settings
from battery_deg_spme.fitting.cycle_pipeline import run_single_cycle_pipeline
from battery_deg_spme.visualization.cycle_plots import (
    plot_cycle_voltage_current,
    plot_selected_cycle,
)

In [ ]:
def summarize_array(name: str, arr):
    if arr is None:
        print(f"{name}: None")
        return

    arr_np = np.asarray(arr)
    print(f"{name}: shape={arr_np.shape}, dtype={arr_np.dtype}")

    if arr_np.size == 0:
        print(f"{name}: empty")
        return

    if np.issubdtype(arr_np.dtype, np.number):
        finite_mask = np.isfinite(arr_np)
        print(
            f"{name}: finite={finite_mask.all()}, "
            f"nan_count={np.isnan(arr_np).sum() if np.issubdtype(arr_np.dtype, np.floating) else 0}, "
            f"min={np.nanmin(arr_np)}, max={np.nanmax(arr_np)}"
        )


def inspect_result_structure(result: dict):
    print("\n================ RESULT STRUCTURE ================\n")
    print("Top-level keys:", list(result.keys()))

    for key in ["cfg", "prep", "proxy", "stage2", "stage3a", "stage3b"]:
        print(f"\n--- {key} ---")
        value = result.get(key, None)

        if value is None:
            print("None")
            continue

        if isinstance(value, dict):
            print("dict keys:", list(value.keys()))
        else:
            print(type(value))

    prep = result.get("prep", {})
    if isinstance(prep, dict):
        print("\n================ PREP SHAPES ================\n")
        for k in ["t", "u", "y"]:
            summarize_array(f"prep['{k}']", prep.get(k, None))

    proxy = result.get("proxy", {})
    if isinstance(proxy, dict):
        print("\n================ PROXY SHAPES ================\n")
        for k in ["X_proxy", "Y_proxy", "U_proxy"]:
            if k in proxy:
                summarize_array(f"proxy['{k}']", proxy.get(k, None))

    for stage_name in ["stage2", "stage3a", "stage3b"]:
        stage = result.get(stage_name, None)
        if not isinstance(stage, dict):
            continue

        print(f"\n================ {stage_name.upper()} SHAPES ================\n")
        for k in [
            "yhat",
            "xhat",
            "thetaZ_hat",
            "theta_lin",
            "theta_hat",
            "P_hat",
            "Q_hat",
            "R_hat",
            "A_hat",
            "B_hat",
            "C_hat",
            "D_hat",
        ]:
            if k in stage:
                summarize_array(f"{stage_name}['{k}']", stage.get(k, None))

        metrics = stage.get("metrics", None)
        if isinstance(metrics, dict):
            print(f"{stage_name} metrics:", metrics)


def inspect_cycle_dataframe(result: dict, settings):
    print("\n================ RAW / SELECTED CYCLE DATA ================\n")

    cycle_df = result.get("cycle_df", None)
    if cycle_df is None:
        print("cycle_df not found in result.")
        return

    print("cycle_df type:", type(cycle_df))
    if hasattr(cycle_df, "shape"):
        print("cycle_df shape:", cycle_df.shape)

    if isinstance(cycle_df, pd.DataFrame):
        print("cycle_df columns:", list(cycle_df.columns))
        display(cycle_df.head())
        display(cycle_df.tail())

        for col in [settings.data.time_col, settings.data.i_col, settings.data.v_col]:
            if col in cycle_df.columns:
                s = pd.to_numeric(cycle_df[col], errors="coerce")
                print(
                    f"{col}: len={len(s)}, nan={s.isna().sum()}, "
                    f"min={s.min()}, max={s.max()}"
                )
            else:
                print(f"{col}: NOT FOUND in cycle_df")

In [ ]:
def run_and_debug_cycle(cycle_index: int):
    print("\n" + "=" * 80)
    print(f"DEBUGGING CYCLE INDEX: {cycle_index}")
    print("=" * 80)

    settings = get_default_settings()
    settings.cycle.cycle_mode = "index"
    settings.cycle.cycle_index = cycle_index

    # Keep notebook/debug mode simple and close to pipeline defaults.
    # Uncomment only if you want to reduce plotting burden.
    # settings.surrogate.nonlinearity_grid_n = 25

    try:
        result = run_single_cycle_pipeline(settings)
        print("\n✅ Pipeline finished successfully.")
        print("Chosen cycle:", result.get("chosen_cycle_idx", None))
        print("Selection note:", result.get("selection_note", None))

        inspect_cycle_dataframe(result, settings)
        inspect_result_structure(result)

        prep = result["prep"]

        plot_selected_cycle(
            cycle_df=result["cycle_df"],
            i_col=settings.data.i_col,
            chosen=result["chosen_cycle_idx"],
            mode=settings.cycle.cycle_mode,
            show=True,
        )

        plot_cycle_voltage_current(
            t=prep["t"],
            u=prep["u"],
            y=prep["y"],
            title=f"Cycle {cycle_index}: initial selected cycle data",
            show=True,
        )

        return result

    except Exception as e:
        print("\n❌ Pipeline failed.")
        print("Exception type:", type(e).__name__)
        print("Exception message:", str(e))
        print("\nFull traceback:\n")
        traceback.print_exc()
        return None

In [ ]:
result_103 = run_and_debug_cycle(103)

In [ ]:
result_104 = run_and_debug_cycle(104)